# Qwen validation on YOLO held-out positives (×3 replicates)

Audit all **115** images that YOLO (`sc_100pct`, `conf=0.01`) called waste-positive on the independent GSVI held-out set:

- **85 TP** (`gt_waste=1`, `pred_waste=1`)
- **30 FP** (`gt_waste=0`, `pred_waste=1`)

Source list (left untouched):
`$PHD_DATA_ROOT/Chapter_waste/3_waste_identification/yolo/heldout_GSVI_p100_image_level_predictions.csv`

This notebook is the **three-replicate** route, for `sc_100pct` only — it is the source of the
replicate-stability result (`table_3_qwen_replicates.csv`). For a **single** pass over **all five**
SC scenarios use `2_qwen_review.py`, which shares this prompt, model and image sources.

What it does:
1. queries the full YOLO-positive set (not only the FPs) — the cascade needs Qwen's verdict on true positives too,
2. runs the full batch **3 independent times**,
3. writes results as columns **`Qwen2_1` / `Qwen2_2` / `Qwen2_3`**.

Outputs:

```
qwen/qwen_replicates_sc_100pct_raw.csv   working table, resumable (180 rows × Qwen2_*)
qwen/qwen_labels_sc_100pct.csv           what 1_heldout_metrics.py reads
```

Rows with `pred_waste=0` keep the Qwen columns blank (Qwen only runs on the 115 YOLO positives).


In [1]:
%pip install -U "segmind>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass

api_key = getpass.getpass("Enter your Segmind API key: ")
os.environ["SEGMIND_API_KEY"] = api_key

print("API key configured.")

API key configured.


## Paths & load the 115 YOLO positives

In [6]:
from __future__ import annotations

import base64
import csv
import os
import re
import time
from pathlib import Path

import segmind

DATA_ROOT = Path(
    os.environ.get(
        "PHD_DATA_ROOT",
        "/Users/wenlanzhang/Downloads/PhD_UCL/Data",
    )
)
IMG_ROOT = DATA_ROOT / "Waste" / "img"
OUTPUT_DIR = DATA_ROOT / "Chapter_waste" / "3_waste_identification"
YOLO_DIR = OUTPUT_DIR / "yolo"
QWEN_DIR = OUTPUT_DIR / "qwen"
SAMPLING_DIR = OUTPUT_DIR / "sampling"

SCENARIO = "sc_100pct"  # this notebook is the sc_100pct 3-replicate route

PRED_CSV = YOLO_DIR / "heldout_GSVI_p100_image_level_predictions.csv"
# Working table with the Qwen columns (the YOLO predictions file is left untouched).
OUT_CSV = QWEN_DIR / f"qwen_replicates_{SCENARIO}_raw.csv"
# Standard label file consumed by 1_heldout_metrics.py.
LABELS_CSV = QWEN_DIR / f"qwen_labels_{SCENARIO}.csv"
INVENTORY_CSV = SAMPLING_DIR / "heldout_GSVI_p100_test.csv"
HELDOUT_IMAGES = (
    IMG_ROOT
    / "7_heldout_validation"
    / "20260807_heldout_GSVI_p100_label"
    / "images"
)

N_REPLICATES = 3
QWEN_COLS = [f"Qwen2_{i}" for i in range(1, N_REPLICATES + 1)]  # Qwen2_1, Qwen2_2, Qwen2_3
SLEEP_S = 0.5
MODEL_ID = "qwen2-vl-72b-instruct"

# PROMPT = """
# Are there any clearly defined piles of household waste (e.g., large amounts of plastic bags, 
# food packaging, paper scraps, beverage bottles, or other domestic waste)? Please answer Yes or No.
# """.strip()

# PROMPT = '''
# Is household waste visible in this street-view image (e.g. plastic bags, food packaging, bottles, 
# paper, cloth, or mixed refuse), whether piled, scattered along the roadside, or dumped near the road? 
# Ignore vegetation, soil, construction materials, and goods stacked for sale. Answer Yes or No only.
# '''.strip()

# PROMPT = '''
# Is household waste visible in this street-view image (plastic bags, packaging, bottles, paper, cloth, 
# food waste, or mixed refuse), even if the amount is modest, scattered, or not in a neat pile?
# Ignore clean road surfaces, vegetation, bare soil, and goods stacked for intentional sale.
# Answer Yes or No only.
# '''.strip()

PROMPT = '''
Is any discarded household refuse visible in this image (bags, packaging, bottles, paper, cloth, 
food waste, or mixed trash)? Count modest or scattered waste as Yes if it is clearly discard/rubbish.
Ignore clean road/sidewalk, vegetation, bare soil, and intentional goods stacked for sale.
Answer Yes or No only.
'''.strip()


assert PRED_CSV.exists(), f"Missing predictions CSV: {PRED_CSV}"
print("DATA_ROOT     :", DATA_ROOT)
print("PRED_CSV      :", PRED_CSV)
print("OUT_CSV       :", OUT_CSV)
print("LABELS_CSV    :", LABELS_CSV)
print("HELDOUT_IMAGES:", HELDOUT_IMAGES, "exists=", HELDOUT_IMAGES.exists())
print("Qwen columns  :", QWEN_COLS)

DATA_ROOT     : /Users/wenlanzhang/Downloads/PhD_UCL/Data
PRED_CSV      : /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/7_heldout_validation/Validation/heldout_GSVI_p100_image_level_predictions.csv
OUT_CSV       : /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/7_heldout_validation/Validation/heldout_GSVI_p100_image_level_predictions_with_qwen.csv
HELDOUT_IMAGES: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/7_heldout_validation/20260807_heldout_GSVI_p100_label/images exists= True
Qwen columns  : ['Qwen2_1', 'Qwen2_2', 'Qwen2_3']


In [8]:
UUID_PREFIX = re.compile(r"^[0-9a-f]{8}-(.+)$", re.I)


def core_stem(name: str) -> str:
    """Strip LabelImg-style leading 8-hex UUID prefixes from a stem/filename."""
    s = Path(name).stem if str(name).lower().endswith((".jpg", ".jpeg", ".png", ".webp")) else str(name)
    while True:
        m = UUID_PREFIX.match(s)
        if not m:
            break
        s = m.group(1)
    return s


def google_path_guesses(core: str) -> list[Path]:
    """GSVI layout: Google/{first}/{second}/{stem}.jpg with case variants."""
    if "_" not in core or len(core) < 2:
        return []
    pan = core.rsplit("_", 1)[0]
    if len(pan) < 2:
        return []
    a, b = pan[0], pan[1]
    out: list[Path] = []
    for aa in {a, a.lower(), a.upper()}:
        for bb in {b, b.lower(), b.upper()}:
            out.append(IMG_ROOT / "Google" / aa / bb / f"{core}.jpg")
    return out


def load_inventory_paths() -> dict[str, Path]:
    mapping: dict[str, Path] = {}
    if not INVENTORY_CSV.exists():
        return mapping
    with INVENTORY_CSV.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            stem = row["img_name"]
            p = IMG_ROOT / row["img_dir"] / f"{stem}.jpg"
            if p.exists():
                mapping[stem] = p
    return mapping


def build_heldout_index() -> dict[str, Path]:
    index: dict[str, Path] = {}
    if not HELDOUT_IMAGES.exists():
        return index
    for p in HELDOUT_IMAGES.glob("*.jpg"):
        index.setdefault(p.stem, p)
        index.setdefault(core_stem(p.stem), p)
    return index


def resolve_image(
    img_name: str,
    heldout_index: dict[str, Path],
    inv_paths: dict[str, Path],
) -> Path | None:
    stem = Path(img_name).stem
    core = core_stem(stem)

    for key in (stem, core):
        if key in heldout_index:
            return heldout_index[key]
        if key in inv_paths:
            return inv_paths[key]

    for p in google_path_guesses(core):
        if p.exists():
            return p

    return None


heldout_index = build_heldout_index()
inv_paths = load_inventory_paths()
print(f"Held-out index: {len(heldout_index)} keys | inventory Google paths: {len(inv_paths)}")

# Prefer resume from OUT_CSV if present; else start from the YOLO predictions.
# The YOLO file covers every scenario and uses its own column names, so filter
# to SCENARIO and map them onto the names this notebook uses.
COLUMN_MAP = {"image_name": "img_name", "ground_truth": "gt_waste", "prediction": "pred_waste"}

source_csv = OUT_CSV if OUT_CSV.exists() else PRED_CSV
with source_csv.open(encoding="utf-8") as f:
    reader = csv.DictReader(f)
    base_fieldnames = list(reader.fieldnames or [])
    table = list(reader)

if source_csv == PRED_CSV:
    table = [r for r in table if r.get("scenario", "").strip() == SCENARIO]
    base_fieldnames = [COLUMN_MAP.get(c, c) for c in base_fieldnames]
    table = [{COLUMN_MAP.get(k, k): v for k, v in r.items()} for r in table]

print(f"Loaded {len(table)} rows from {source_csv.name} (scenario={SCENARIO})")

for col in QWEN_COLS:
    if col not in base_fieldnames:
        base_fieldnames.append(col)
for row in table:
    for col in QWEN_COLS:
        row.setdefault(col, "")

yolo_pos_idx = [
    i for i, r in enumerate(table) if str(r.get("pred_waste", "")).strip() == "1"
]
assert len(yolo_pos_idx) == 115, f"Expected 115 YOLO positives, got {len(yolo_pos_idx)}"

missing: list[str] = []
path_by_img: dict[str, Path] = {}
for i in yolo_pos_idx:
    img_name = table[i]["img_name"]
    path = resolve_image(img_name, heldout_index, inv_paths)
    if path is None:
        missing.append(img_name)
    else:
        path_by_img[img_name] = path.resolve()

records = sorted(
    [
        {
            "row_idx": i,
            "filename": table[i]["img_name"],
            "gt_waste": int(table[i].get("gt_waste", 0)),
            "filepath": path_by_img[table[i]["img_name"]],
        }
        for i in yolo_pos_idx
        if table[i]["img_name"] in path_by_img
    ],
    key=lambda r: r["filename"],
)

n_tp = sum(1 for r in records if r["gt_waste"] == 1)
n_fp = sum(1 for r in records if r["gt_waste"] == 0)
print(f"YOLO positives resolved: {len(records)} / 115  (TP={n_tp}, FP={n_fp})")
if missing:
    print(f"WARNING: {len(missing)} image(s) not found on disk:")
    for name in missing:
        print("  -", name)
else:
    print("All 115 images located.")

assert len(records) == 115, (
    f"Need all 115 images before running Qwen; missing {len(missing)}. "
    "Check HELDOUT_IMAGES / PHD_DATA_ROOT / Google tree."
)


def save_table() -> None:
    """Write full 180-row table (+ Qwen2_* columns) to OUT_CSV."""
    QWEN_DIR.mkdir(parents=True, exist_ok=True)
    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=base_fieldnames, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(table)

Held-out index: 180 keys | inventory Google paths: 180
Loaded 180 rows from heldout_GSVI_p100_image_level_predictions.csv
YOLO positives resolved: 115 / 115  (TP=85, FP=30)
All 115 images located.


## Helpers + smoke test (1 image)

In [9]:
def image_to_data_url(image_path: Path) -> str:
    suffix = image_path.suffix.lower()
    mime_types = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".webp": "image/webp",
    }
    if suffix not in mime_types:
        raise ValueError(f"Unsupported image type: {suffix}")
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_types[suffix]};base64,{encoded}"


def qwen_yes_no(image_path: Path) -> str:
    image_data = image_to_data_url(image_path)
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": PROMPT},
                    {"type": "image_url", "image_url": {"url": image_data}},
                ],
            }
        ]
    }
    reply = segmind.chat(MODEL_ID, **payload)
    text = getattr(reply, "text", None)
    if text is None:
        text = str(reply) if reply is not None else ""
    return (text or "").strip()


test = records[0]
print("Testing:", test["filename"])
print("Path   :", test["filepath"])
print("gt_waste=", test["gt_waste"], "(YOLO pred_waste=1)")
print("Qwen  :", qwen_yes_no(test["filepath"]))


Testing: 03289dd4-HftpqnG8JrJHKaAAzGphWQ_0.jpg
Path   : /Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/img/7_heldout_validation/20260807_heldout_GSVI_p100_label/images/HftpqnG8JrJHKaAAzGphWQ_0.jpg
gt_waste= 1 (YOLO pred_waste=1)
Qwen  : Yes.


## Run all 115 images × 3 replicates

Fills columns **`Qwen2_1` / `Qwen2_2` / `Qwen2_3`** on `OUT_CSV` (new version of the predictions table).

- Only rows with `pred_waste=1` are queried.
- After each replicate, **retry any missing/empty/error cells** until that column is complete, then move to the next replicate.
- Progress is saved after every image (safe to interrupt / resume).
- Re-run a single replicate with e.g. `RUN_REPLICATES = [2]`.
- `OVERWRITE_EXISTING = True` forces re-query of already-filled cells.


In [10]:
RUN_REPLICATES = list(range(1, N_REPLICATES + 1))  # e.g. [1] or [2, 3]
OVERWRITE_EXISTING = False  # True = re-query even if Qwen2_k is already filled
MAX_RETRY_ROUNDS = 20       # safety cap per missing cell campaign (per replicate)
RETRY_SLEEP_S = 1.5         # slightly longer pause between retry attempts


def is_missing_response(val) -> bool:
    """Treat blank / NaN / ERROR rows as incomplete (need re-query)."""
    if val is None:
        return True
    s = str(val).strip()
    if not s:
        return True
    if s.lower() in {"nan", "none", "null", "na"}:
        return True
    if s.upper().startswith("ERROR"):
        return True
    return False


def missing_for_col(col: str) -> list[dict]:
    out = []
    for rec in records:
        row = table[rec["row_idx"]]
        if OVERWRITE_EXISTING or is_missing_response(row.get(col, "")):
            out.append(rec)
    return out


def query_and_fill(rec: dict, col: str, tag: str) -> bool:
    """Call Qwen; return True if we stored a non-empty answer."""
    print(f"{tag} {rec['filename']} ...", end=" ", flush=True)
    try:
        answer = qwen_yes_no(rec["filepath"])
        if is_missing_response(answer):
            table[rec["row_idx"]][col] = ""
            print("✗ empty API response")
            return False
        table[rec["row_idx"]][col] = answer
        print("✓", (answer or "")[:40].replace("\n", " "))
        return True
    except Exception as e:
        # leave empty so the retry loop picks it up again
        table[rec["row_idx"]][col] = ""
        print(f"✗ {e}")
        return False
    finally:
        save_table()
        time.sleep(SLEEP_S)


def run_replicate(rep: int) -> None:
    col = f"Qwen2_{rep}"
    assert col in QWEN_COLS
    print(f"\n======== Replicate {rep}/{N_REPLICATES} → column {col} ========")

    # --- pass 1: full set (or only missing if resuming) ---
    todo = missing_for_col(col)
    n_ok = n_err = 0
    for i, rec in enumerate(todo, start=1):
        ok = query_and_fill(rec, col, f"[rep{rep} {i}/{len(todo)}]")
        n_ok += int(ok)
        n_err += int(not ok)
    print(f"Initial pass: success={n_ok}, failed/empty={n_err}")

    # --- keep retrying missing cells until none remain ---
    for round_i in range(1, MAX_RETRY_ROUNDS + 1):
        still = missing_for_col(col)
        # if OVERWRITE_EXISTING, missing_for_col returns everyone — only retry blanks
        still = [
            rec
            for rec in still
            if is_missing_response(table[rec["row_idx"]].get(col, ""))
        ]
        if not still:
            print(f"Column {col} complete: no missing cells among {len(records)} YOLO positives.")
            break

        print(
            f"\n--- Retry round {round_i}/{MAX_RETRY_ROUNDS}: "
            f"{len(still)} missing cell(s) in {col} ---"
        )
        for i, rec in enumerate(still, start=1):
            query_and_fill(rec, col, f"[retry{round_i} {i}/{len(still)}]")
            time.sleep(max(0.0, RETRY_SLEEP_S - SLEEP_S))  # extra backoff on retries
    else:
        still = [
            rec
            for rec in records
            if is_missing_response(table[rec["row_idx"]].get(col, ""))
        ]
        if still:
            names = [r["filename"] for r in still[:10]]
            raise RuntimeError(
                f"Column {col} still has {len(still)} missing after "
                f"{MAX_RETRY_ROUNDS} retry rounds. Example: {names}"
            )

    n_filled = sum(
        1
        for rec in records
        if not is_missing_response(table[rec["row_idx"]].get(col, ""))
    )
    print(f"Replicate {rep} done: filled={n_filled}/{len(records)} → {OUT_CSV}")


save_table()
print(f"Working file: {OUT_CSV}")

for rep in RUN_REPLICATES:
    run_replicate(int(rep))

print("\nFinished. Image-level predictions + Qwen replicates saved to:")
print(" ", OUT_CSV)

# final completeness check
for col in QWEN_COLS:
    if col not in [f"Qwen2_{r}" for r in RUN_REPLICATES]:
        continue
    n_miss = sum(
        1
        for rec in records
        if is_missing_response(table[rec["row_idx"]].get(col, ""))
    )
    print(f"  {col}: missing={n_miss}/115")


Working file: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/7_heldout_validation/Validation/heldout_GSVI_p100_image_level_predictions_with_qwen.csv

======== Replicate 1/3 → column Qwen2_1 ========
[rep1 1/115] 03289dd4-HftpqnG8JrJHKaAAzGphWQ_0.jpg ... ✓ Yes
[rep1 2/115] 033e2fe7-MCkkOZ6Ow8oUlqi9tyv6HQ_90.jpg ... ✓ Yes
[rep1 3/115] 0539e2ea-6qxxsCdr7raOL7ecdadXdA_270.jpg ... ✓ Yes.
[rep1 4/115] 05b0b5f4-sg5W4wNTCpbKexyvYnnPuA_270.jpg ... ✓ Yes
[rep1 5/115] 070c5c3d-Jyhy2ROxqxt-Rqzgw6Q4jQ_180.jpg ... ✓ Yes.
[rep1 6/115] 076390bd-Il6X4MJDC0odhEWTJGUK3w_180.jpg ... ✓ Yes.
[rep1 7/115] 0o5SkIs7bA6N8GlGtD4jKQ_90.jpg ... ✓ No
[rep1 8/115] 1002b381-WcVQJHy4AE-U-xkZ8zYzQA_90.jpg ... ✓ Yes
[rep1 9/115] 134b71ae-gr4cY2J_DlHzjXmRK1dXWw_0.jpg ... ✓ Yes
[rep1 10/115] 13717d98-RegSP6trLsjYKF9ZnZcK6g_90.jpg ... ✓ Yes
[rep1 11/115] 1JNhbluOQCy_rje7QQfMQg_270.jpg ... ✓ No
[rep1 12/115] 1ea5c064-GXz3KhpkSxhcKGr2iGFnvg_270.jpg ... ✓ Yes
[rep1 13/115] 1f1e8ebe-0Ra1HqXU7NIqTuuYOpflXw_0.jpg ... ✓ 

## Quick agreement check on `Qwen2_1..3`

Reads the written OUT_CSV and prints yes/no counts + unanimity among YOLO positives.

In [11]:
from collections import Counter


def parse_yes_no(text: str) -> str | None:
    if text is None:
        return None
    t = str(text).strip().lower()
    if not t or t.startswith("error"):
        return None
    first = re.split(r"[\s,.;:]+", t, maxsplit=1)[0]
    if first in {"yes", "y"}:
        return "yes"
    if first in {"no", "n"}:
        return "no"
    if t.startswith("yes"):
        return "yes"
    if t.startswith("no"):
        return "no"
    return None


with OUT_CSV.open(encoding="utf-8") as f:
    out_rows = list(csv.DictReader(f))

pos_rows = [r for r in out_rows if str(r.get("pred_waste", "")).strip() == "1"]
print(f"OUT_CSV: {OUT_CSV.name}  rows={len(out_rows)}  YOLO pos={len(pos_rows)}")

for col in QWEN_COLS:
    labels = [parse_yes_no(r.get(col, "")) for r in pos_rows]
    filled = sum(1 for v in labels if v is not None)
    print(f"  {col}: filled={filled}/115  counts={dict(Counter(labels))}")

n_unanimous = 0
maj_counts: Counter = Counter()
for r in pos_rows:
    labs = [parse_yes_no(r.get(c, "")) for c in QWEN_COLS]
    labs = [x for x in labs if x is not None]
    if len(labs) == N_REPLICATES and len(set(labs)) == 1:
        n_unanimous += 1
        maj_counts[labs[0]] += 1
    elif labs:
        top = Counter(labs).most_common()
        if len(top) == 1 or top[0][1] > top[1][1]:
            maj_counts[top[0][0]] += 1
        else:
            maj_counts["(tie)"] += 1
    else:
        maj_counts["(empty)"] += 1

print(f"Unanimous {N_REPLICATES}/{N_REPLICATES}: {n_unanimous}/115")
print(f"Majority label among YOLO positives: {dict(maj_counts)}")
print(f"Table path: {OUT_CSV}")

OUT_CSV: heldout_GSVI_p100_image_level_predictions_with_qwen.csv  rows=180  YOLO pos=115
  Qwen2_1: filled=115/115  counts={'yes': 85, 'no': 30}
  Qwen2_2: filled=115/115  counts={'yes': 87, 'no': 28}
  Qwen2_3: filled=115/115  counts={'yes': 88, 'no': 27}
Unanimous 3/3: 108/115
Majority label among YOLO positives: {'yes': 87, 'no': 28}
Table path: /Users/wenlanzhang/Downloads/PhD_UCL/Data/Chapter_waste/7_heldout_validation/Validation/heldout_GSVI_p100_image_level_predictions_with_qwen.csv


In [ ]:
# Export the standard label file that 1_heldout_metrics.py reads.
# Only the YOLO positives carry Qwen answers, so only those rows are written.
QWEN_DIR.mkdir(parents=True, exist_ok=True)

label_rows = [
    {
        "image_name": r["img_name"],
        "scenario": SCENARIO,
        **{c: r.get(c, "") for c in QWEN_COLS},
    }
    for r in table
    if str(r.get("pred_waste", "")).strip() == "1"
]
label_rows.sort(key=lambda r: r["image_name"])

# Guard: never overwrite a good label file with blanks. Running this cell before
# the replicates have been filled in would otherwise wipe the existing answers.
def _blank(v) -> bool:
    s = "" if v is None else str(v).strip()
    return not s or s.lower() in {"nan", "none", "null", "na"} or s.upper().startswith("ERROR")


n_answered = sum(1 for r in label_rows if any(not _blank(r.get(c, "")) for c in QWEN_COLS))
if n_answered < len(label_rows):
    raise RuntimeError(
        f"Only {n_answered}/{len(label_rows)} YOLO positives have a Qwen answer. "
        f"Run the replicate cell above first; refusing to overwrite {LABELS_CSV.name}."
    )

with LABELS_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_name", "scenario", *QWEN_COLS])
    writer.writeheader()
    writer.writerows(label_rows)

print(f"Wrote {LABELS_CSV}  ({len(label_rows)} YOLO-positive rows)")
print("Next: python 3_waste_identification/1_heldout_metrics.py")
